# Clickbait Detection — Transformer Fine-Tuning

**Projekt:** Clickbait Headline Detection using BERT and RoBERTa
**Kurs:** Übung Informationslinguistik 2 (NLE 2), Universität Regensburg
**Autoren:** Rustem Kuduschev (2184638), Tim Frummet (2388999)

Dieses Notebook fine-tuned **BERT**, **RoBERTa** und **ModernBERT** auf den
Webis-Clickbait-Corpus 2017 und vergleicht sie mit der klassischen TF-IDF-Baseline.

---

## Vorbereitung: GPU aktivieren

**Wichtig:** Oben im Menü → *Laufzeit* → *Laufzeittyp ändern* → **T4 GPU** auswählen.
Ohne GPU dauert ein Trainingslauf statt ~10 Minuten mehrere Stunden.

Die nächste Zelle prüft, ob die GPU aktiv ist.

In [3]:
!nvidia-smi

Thu Sep 17 14:20:13 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   37C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 1. Pakete installieren

ModernBERT benötigt `transformers>=4.48`. Nach der Installation muss die Laufzeit
ggf. neu gestartet werden (Colab zeigt einen Button an — dann einfach ab Zelle 2
weiterlaufen lassen).

In [4]:
!pip install -q -U "transformers>=4.48" datasets accelerate scikit-learn
print("Fertig. Falls Colab einen Neustart anbietet: neu starten und ab Zelle 2 weiter.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 70.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 23.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 71.9 MB/s eta 0:00:00
Fertig. Falls Colab einen Neustart anbietet: neu starten und ab Zelle 2 weiter.


## 2. Daten hochladen

Benötigt werden die drei CSV-Dateien, die `src/data_prep.py` lokal erzeugt:
`train.csv`, `val.csv`, `test.csv` (im Projektordner unter `data/`).

Im Upload-Dialog alle drei Dateien auswählen. Der Upload ist pro Colab-Session
einmal erforderlich.

Die CSVs werden lokal mit `src/data_prep.py` erzeugt (`seed=42`, stratifizierter
70/15/15-Split) aus dem Webis-Clickbait-Corpus 2017.

In [5]:
from google.colab import files
uploaded = files.upload()   # train.csv, val.csv, test.csv auswählen

Saving test.csv to test.csv
Saving train.csv to train.csv
Saving val.csv to val.csv


## 3. Daten laden und prüfen

Kontrolle: Der Clickbait-Anteil sollte in allen Splits bei ~0.24 liegen.

In [6]:
import pandas as pd

DATA_DIR = ''   # Dateien liegen nach dem Upload direkt im Arbeitsverzeichnis

train_df = pd.read_csv(DATA_DIR + 'train.csv')
val_df   = pd.read_csv(DATA_DIR + 'val.csv')
test_df  = pd.read_csv(DATA_DIR + 'test.csv')

for name, df in [('train', train_df), ('val', val_df), ('test', test_df)]:
    ratio = df['label'].mean()
    print(f"{name:6s} n={len(df):6d}  clickbait_ratio={ratio:.4f}")

train_df.head()

train  n= 13638  clickbait_ratio=0.2420
val    n=  2923  clickbait_ratio=0.2422
test   n=  2923  clickbait_ratio=0.2419


,id,headline,target_title,truth_mean,label
0,856707338882785280,#AnzacDay marked by Australian Antarctic exped...,Anzac Day marked by Australian Antarctic exped...,0.000000,0
1,841627477491441664,Should we be allowed to use superannuation on ...,Should we be allowed to use superannuation on ...,0.466667,1
2,852110890057035778,This 85-year-old wants to become the oldest pe...,"Mount Everest: Nepali Man, 85, Aims to Become ...",0.133333,0
3,824672023012114432,Will Matt Ryan's redemption season end with a ...,Can Matt Ryan ride this season to a Lombardi t...,0.133333,0
4,845005253338681344,An exasperated Samantha Bee thinks Trump's pro...,Samantha Bee: Trump's budget reveals he 'has n...,0.266667,0


## 4. Setup: Seeds, Metriken, gewichteter Loss

**Reproduzierbarkeit:** Alle Zufallsprozesse werden über `SEED = 42` fixiert.

**Gewichteter Loss:** Bei ~24 % Clickbait würde ein unbalanciertes Training die
Minderheitsklasse vernachlässigen. Der `WeightedTrainer` gewichtet den Loss
invers zur Klassenhäufigkeit — das Gegenstück zu `class_weight='balanced'`
in der Baseline.

In [7]:
import os, random, time, json
import numpy as np
import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, DataCollatorWithPadding, set_seed
)
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    classification_report, confusion_matrix
)

SEED = 42

def seed_everything(seed=SEED):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    set_seed(seed)

seed_everything()
print("Device:", "cuda" if torch.cuda.is_available() else "CPU (!) — GPU aktivieren!")

Device: cuda


In [8]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    p, r, f1, _ = precision_recall_fscore_support(
        labels, preds, average='binary', zero_division=0)
    return {'accuracy': accuracy_score(labels, preds),
            'precision': p, 'recall': r, 'f1': f1}


class WeightedTrainer(Trainer):
    """Trainer mit klassengewichtetem Loss gegen das Ungleichgewicht (~24% Clickbait)."""
    def __init__(self, class_weights=None, **kwargs):
        super().__init__(**kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop('labels')
        outputs = model(**inputs)
        logits = outputs.logits
        w = self.class_weights.to(logits.device) if self.class_weights is not None else None
        loss = torch.nn.CrossEntropyLoss(weight=w)(logits, labels)
        return (loss, outputs) if return_outputs else loss


def get_class_weights(df):
    n = len(df); n_pos = int(df['label'].sum()); n_neg = n - n_pos
    return torch.tensor([n / (2 * n_neg), n / (2 * n_pos)], dtype=torch.float)

print("Klassengewichte:", get_class_weights(train_df).tolist())

Klassengewichte: [0.6596691608428955, 2.065737724304199]


## 5. Die zentrale Trainingsfunktion

Diese Funktion trainiert jedes der drei Modelle; es wird nur der `model_name`
gewechselt. Die `Auto*`-Klassen von Hugging Face laden den zum Modell passenden
Tokenizer automatisch.

**Hyperparameter:**
- `learning_rate=2e-5` — Standardbereich für Transformer-Fine-Tuning (1e-5 bis 5e-5).
  Höhere Werte führen zu *catastrophic forgetting*: das Modell verliert sein Pretraining-Wissen.
- `max_length=64` — Headlines sind kurz; der Default von 512 wäre reine Rechenverschwendung.
- `epochs=3` — bei Fine-Tuning reichen wenige Epochen; mehr führt schnell zu Overfitting.
- `metric_for_best_model='f1'` — nicht Accuracy, weil die bei 24 % Positivrate irreführt.

In [9]:
def run_experiment(model_name, train_df, val_df, test_df,
                   epochs=3, batch_size=16, lr=2e-5, max_length=64,
                   use_class_weights=True, seed=SEED):
    seed_everything(seed)
    short = model_name.split('/')[-1]
    print(f"\n{'='*60}\n  {model_name}\n{'='*60}")

    tokenizer = AutoTokenizer.from_pretrained(model_name)

    def tokenize(batch):
        return tokenizer(batch['headline'], truncation=True, max_length=max_length)

    def to_ds(df):
        ds = Dataset.from_pandas(df[['headline', 'label']], preserve_index=False)
        return ds.map(tokenize, batched=True)

    train_ds, val_ds, test_ds = to_ds(train_df), to_ds(val_df), to_ds(test_df)

    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

    args = TrainingArguments(
        output_dir=f'./results/{short}',
        num_train_epochs=epochs,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=64,
        learning_rate=lr,
        weight_decay=0.01,
        eval_strategy='epoch',
        save_strategy='epoch',
        load_best_model_at_end=True,
        metric_for_best_model='f1',
        greater_is_better=True,
        save_total_limit=1,
        seed=seed,
        logging_steps=100,
        report_to='none',
        fp16=torch.cuda.is_available(),
    )

    trainer = WeightedTrainer(
        class_weights=get_class_weights(train_df) if use_class_weights else None,
        model=model, args=args,
        train_dataset=train_ds, eval_dataset=val_ds,
        compute_metrics=compute_metrics,
        data_collator=DataCollatorWithPadding(tokenizer),
    )

    # Trainingsaufwand als Vergleichsdimension messen
    t0 = time.time()
    trainer.train()
    train_time = time.time() - t0

    test_metrics = trainer.evaluate(test_ds, metric_key_prefix='test')
    preds = np.argmax(trainer.predict(test_ds).predictions, axis=1)

    n_params = sum(p.numel() for p in model.parameters())
    result = {
        'model': model_name,
        'test_f1': test_metrics['test_f1'],
        'test_precision': test_metrics['test_precision'],
        'test_recall': test_metrics['test_recall'],
        'test_accuracy': test_metrics['test_accuracy'],
        'train_time_sec': round(train_time, 1),
        'n_params_mio': round(n_params / 1e6, 1),
    }

    print(f"\n--- {short} ---")
    print(classification_report(test_df['label'], preds,
          target_names=['kein Clickbait', 'Clickbait'], digits=3))
    print("Konfusionsmatrix:\n", confusion_matrix(test_df['label'], preds))
    print(f"Trainingszeit: {train_time:.1f}s | Parameter: {n_params/1e6:.1f} Mio")

    return result, preds

## 6. Training: BERT

Auf einer T4-GPU dauert ein Durchlauf ungefähr 4–5 Minuten.

Für einen schnellen Funktionstest kann `train_df` durch
`train_df.sample(1000, random_state=42)` ersetzt werden.

In [10]:
results = {}
predictions = {}

results['bert'], predictions['bert'] = run_experiment(
    'bert-base-uncased', train_df, val_df, test_df)


  bert-base-uncased


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.atte

Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,F1
0.212406,0.508272,3,0.845706,0.668865,0.717115,0.692150



--- bert-base-uncased ---
                precision    recall  f1-score   support

kein Clickbait      0.908     0.887     0.897      2216
     Clickbait      0.669     0.717     0.692       707

      accuracy                          0.846      2923
     macro avg      0.788     0.802     0.795      2923
  weighted avg      0.850     0.846     0.847      2923

Konfusionsmatrix:
 [[1965  251]
 [ 200  507]]
Trainingszeit: 211.1s | Parameter: 109.5 Mio


## 7. Training: RoBERTa

Gleiche Funktion, nur anderer Modellname. RoBERTa nutzt BPE statt WordPiece —
der passende Tokenizer wird automatisch geladen.

In [11]:
results['roberta'], predictions['roberta'] = run_experiment(
    'roberta-base', train_df, val_df, test_df)


  roberta-base


config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,F1
0.269663,0.583475,3,0.844680,0.667107,0.714286,0.689891



--- roberta-base ---
                precision    recall  f1-score   support

kein Clickbait      0.907     0.886     0.896      2216
     Clickbait      0.667     0.714     0.690       707

      accuracy                          0.845      2923
     macro avg      0.787     0.800     0.793      2923
  weighted avg      0.849     0.845     0.846      2923

Konfusionsmatrix:
 [[1964  252]
 [ 202  505]]
Trainingszeit: 255.2s | Parameter: 124.6 Mio


## 8. Training: ModernBERT

ModernBERT (2024) ist die modernisierte Encoder-Architektur mit längerem
Kontextfenster, effizienterer Attention und aktuellerem Pretraining-Korpus.

Falls ein Fehler wegen der Transformers-Version auftritt: Zelle 1 erneut ausführen
und Laufzeit neu starten.

In [12]:
results['modernbert'], predictions['modernbert'] = run_experiment(
    'answerdotai/ModernBERT-base', train_df, val_df, test_df)


  answerdotai/ModernBERT-base


config.json:   0%|          | 0.00/1.19k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/20.8k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.13M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,F1
0.146223,0.529127,3,0.849812,0.683562,0.705799,0.694502



--- ModernBERT-base ---
                precision    recall  f1-score   support

kein Clickbait      0.905     0.896     0.900      2216
     Clickbait      0.684     0.706     0.695       707

      accuracy                          0.850      2923
     macro avg      0.794     0.801     0.797      2923
  weighted avg      0.852     0.850     0.851      2923

Konfusionsmatrix:
 [[1985  231]
 [ 208  499]]
Trainingszeit: 355.7s | Parameter: 149.6 Mio


## 9. Vergleichstabelle

Die Baseline-Werte stammen aus dem lokalen Lauf von `baseline.py`.

In [13]:
# Baseline-Werte aus dem lokalen Lauf (baseline.py)
baseline_row = {
    'model': 'TF-IDF + LinearSVM (Baseline)',
    'test_f1': 0.543, 'test_precision': 0.485,
    'test_recall': 0.617, 'test_accuracy': 0.749,
    'train_time_sec': None, 'n_params_mio': None,
}

comparison = pd.DataFrame([baseline_row] + list(results.values()))
comparison = comparison[['model', 'test_precision', 'test_recall',
                         'test_f1', 'test_accuracy',
                         'train_time_sec', 'n_params_mio']]
comparison = comparison.round(3)
display(comparison)

comparison.to_csv('model_comparison.csv', index=False)
print("\nAls model_comparison.csv gespeichert.")

,model,test_precision,test_recall,test_f1,test_accuracy,train_time_sec,n_params_mio
0,TF-IDF + LinearSVM (Baseline),0.485,0.617,0.543,0.749,NaN,NaN
1,bert-base-uncased,0.669,0.717,0.692,0.846,211.1,109.5
2,roberta-base,0.667,0.714,0.690,0.845,255.2,124.6
3,answerdotai/ModernBERT-base,0.684,0.706,0.695,0.850,355.7,149.6



Als model_comparison.csv gespeichert.


## 10. Fehleranalyse

Qualitative Analyse der Fehlklassifikationen: Verteilung nach Clickbait-Muster
und konkrete Beispiele für False Positives und False Negatives.

In [19]:
import re

def categorize(headline):
    h = str(headline).lower()
    if re.match(r'^\s*\d+\s', str(headline)) or re.search(r'\b\d+\s+(reasons|things|ways|photos)\b', h):
        return 'listicle'
    if '?' in str(headline):
        return 'rhetorical_question'
    if any(w in h.split() for w in ['you', 'your', "you'll", "you're"]):
        return 'direct_address'
    if any(w in h for w in ['this', 'these', 'what', 'why']):
        return 'curiosity_gap'
    return 'other'


def analyze_errors(model_key, n_examples=5):
    df = test_df.copy().reset_index(drop=True)
    df['pred'] = predictions[model_key]
    err = df[df['label'] != df['pred']].copy()
    err['error_type'] = np.where(err['pred'] == 1, 'false_positive', 'false_negative')
    err['pattern'] = err['headline'].apply(categorize)

    print(f"\n=== Fehleranalyse: {model_key} ===")
    print(f"Fehler gesamt: {len(err)} von {len(df)} ({len(err)/len(df)*100:.1f}%)\n")
    print(err.groupby(['pattern', 'error_type']).size().unstack(fill_value=0))

    for et in ['false_positive', 'false_negative']:
        subset = err[err['error_type'] == et]
        print(f"\n--- Beispiele: {et} ({len(subset)}) ---")
        for h in subset['headline'].head(n_examples):
            print(f"  • {h}")
    return err

errors_best = analyze_errors('roberta')   # ggf. auf das beste Modell anpassen


=== Fehleranalyse: roberta ===
Fehler gesamt: 454 von 2923 (15.5%)

error_type           false_negative  false_positive
pattern                                            
curiosity_gap                    24              53
direct_address                   13              20
listicle                          3              21
other                           145             126
rhetorical_question              17              32

--- Beispiele: false_positive (252) ---
  • She’s 59 and fit to jump from an airplane
  • The Trump administration banned laptops, so this airline is passing them out
  • The biggest #OscarNoms snubs and surprises  via @TODAYshow
  • Are you on track to reach your retirement goals? Find out:
  • 21 things you know if you live paycheck to paycheck

--- Beispiele: false_negative (202) ---
  • Introducing our first-ever tally of America's wealthiest celebrities:
  • An anti-Trump movement is calling for the boycott of these 36 DAPL-linked banks
  • The new "Beaut

## 11. Ergebnisse sichern

Exportiert Vergleichstabelle, Metriken und Fehleranalyse als Dateien.

In [20]:
with open('results_summary.json', 'w') as f:
    json.dump(results, f, indent=2)

errors_best.to_csv('error_analysis.csv', index=False)

from google.colab import files
files.download('model_comparison.csv')
files.download('results_summary.json')
files.download('error_analysis.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

---

## Mehrere Seeds für statistische Aussagekraft

Liegen die Modelle nah beieinander, ist ein Einzellauf pro Modell nicht aussagekräftig.
Drei Seeds pro Modell liefern Mittelwert und Standardabweichung, sodass sich beurteilen
lässt, ob die Unterschiede innerhalb der Streuung liegen.

Laufzeit: etwa 45–60 Minuten für sechs zusätzliche Läufe.

In [16]:
seeds = [42, 1337, 2024]
model_names = {
    'bert': 'bert-base-uncased',
    'roberta': 'roberta-base',
    'modernbert': 'answerdotai/ModernBERT-base',
}

multi = []
for key, mname in model_names.items():
    for s in seeds:
        if s == 42 and key in results:
            r = dict(results[key]); r['seed'] = s; r['key'] = key
            multi.append(r)
            continue
        r, _ = run_experiment(mname, train_df, val_df, test_df, seed=s)
        r['seed'] = s; r['key'] = key
        multi.append(r)

multi_df = pd.DataFrame(multi)
multi_df.to_csv('multi_seed_raw.csv', index=False)

summary = multi_df.groupby('key')['test_f1'].agg(['mean', 'std', 'min', 'max']).round(4)
print(summary)

for key, row in summary.iterrows():
    print(f"{key:12s} F1 = {row['mean']:.3f} ± {row['std']:.3f}")


  bert-base-uncased


Map:   0%|          | 0/13638 [00:00<?, ? examples/s]

Map:   0%|          | 0/2923 [00:00<?, ? examples/s]

Map:   0%|          | 0/2923 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,F1
0.190861,0.418483,3,0.839206,0.632402,0.800566,0.706617



--- ModernBERT-base ---
                precision    recall  f1-score   support

kein Clickbait      0.930     0.852     0.889      2216
     Clickbait      0.632     0.801     0.707       707

      accuracy                          0.839      2923
     macro avg      0.781     0.826     0.798      2923
  weighted avg      0.858     0.839     0.845      2923

Konfusionsmatrix:
 [[1887  329]
 [ 141  566]]
Trainingszeit: 469.3s | Parameter: 149.6 Mio
              mean     std     min     max
key                                       
bert        0.6884  0.0037  0.6848  0.6922
modernbert  0.7018  0.0065  0.6945  0.7066
roberta     0.7014  0.0131  0.6899  0.7157
bert         F1 = 0.688 ± 0.004
modernbert   F1 = 0.702 ± 0.006
roberta      F1 = 0.701 ± 0.013


### Export der Multi-Seed-Ergebnisse

In [17]:
import os
print([f for f in os.listdir('.') if f.endswith('.csv') or f.endswith('.json')])

['test.csv', 'model_comparison.csv', 'val.csv', 'multi_seed_raw.csv', 'results_summary.json', 'train.csv', 'error_analysis.csv']


In [21]:
from google.colab import files
summary.to_csv('multi_seed_summary.csv')
files.download('multi_seed_raw.csv')
files.download('multi_seed_summary.csv')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

---

## Dokumentierte Parameter

- **Splits:** 70/15/15, stratifiziert, `seed=42`, erzeugt mit `src/data_prep.py`
- **Label:** binär über `truthClass` (Mehrheitsvotum der fünf Annotator:innen)
- **Klassengewichte:** invers zur Klassenhäufigkeit (0,66 / 2,07)
- **Hyperparameter:** lr 2e-5, 3 Epochen, batch size 16, max_length 64
- **Modellauswahl:** bester Checkpoint nach Validierungs-F1
- **Hardware:** Google Colab, NVIDIA Tesla T4